# Built-in fiber generators

TANGLE provides small crossing generators for mechanics tests and a
configurable population generator for material recipes. Generation is
seeded and deterministic; relaxation is tolerance-deterministic rather
than promised bitwise-identical across parallel backends. Every
generator takes a `material=` for fiber name, diameter, and bend limit.

In [ ]:
import math

import tangle
from tangle.units import mm, um

# periodic="xy" makes z the stack axis for layered populations below.
cell = tangle.Cell([1 * mm, 1 * mm, 1 * mm], periodic="xy")
large = tangle.Material("large", diameter=19 * um)
bend_limited = tangle.Material("bend-limited", diameter=19 * um, min_bend_radius=50 * um)

# Point crossings are the minimal rigid-contact demonstration.
point_crossing = tangle.generate_point_crossing(
    cell, material=large, count=8, length=0.8 * mm, name="center crossing",
)
# Distinct placed/rest shape controls create initially bent fibers.
curved_crossing = tangle.generate_multisegment_crossing(
    cell, material=bend_limited, count=4, segments_per_fiber=8,
    length=0.8 * mm, placed_chord_fraction=0.8,
    rest_shape="straight", rest_amplitude=0.0,
    placed_shape="curved", placed_amplitude=0.1 * mm,
)
# A two-fiber pair is useful for isolated refinement/contact tests.
pair = tangle.generate_fiber_pair_crossing(
    cell, material=large, segments_per_fiber=1, length=0.8 * mm,
    axis_separation=10 * um, crossing_angle_degrees=90.0,
)
[len(point_crossing), len(curved_crossing), len(pair)]

## Every `FiberPopulation` field

`FiberPopulation` describes a seeded population with keyword
arguments. Length-like fields accept a single value or a `(min, max)`
tuple that is sampled uniformly. The defaults below are read from the
compiled extension.

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `material` | Material assigned to every generated fiber (name, contact diameter, bend limit). | `Material` |
| `count` | Number of fibers to generate. | count |
| `segments_per_fiber` | Initial uniform centerline resolution. | count |
| `seed` | Deterministic sampling seed. | integer |
| `length` | Fiber length: one value, or a `(min, max)` uniform range. | m or `(m, m)` |
| `diameter` | Sampled contact diameter; `None` uses the material's diameter. | m, `(m, m)`, or `None` |
| `curvature_amplitude` | Generated waviness amplitude. | m or `(m, m)` |
| `nominal_parent_length` | Optional physical parent length for periodic fragments. | m or `None` |
| `orientation` | Orientation distribution object (see the next table). | `IsotropicOrientation`, `PlanarOrientation`, `LayeredBiaxialOrientation`, `AlignedOrientation` |
| `position` | Center-position distribution object (see below). | `UniformPosition`, `LayeredPosition`, `DensityGradientPosition` |
| `max_attempts_per_fiber` | Rejection-sampling budget per fiber. | count |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
population = tangle.FiberPopulation()
fields = ['material', 'count', 'segments_per_fiber', 'seed', 'length', 'diameter', 'curvature_amplitude', 'nominal_parent_length', 'orientation', 'position', 'max_attempts_per_fiber']
{name: getattr(population, name) for name in fields}

## Orientation and position distributions

Each distribution is its own class, so its options are keyword
arguments of that class rather than loose fields that only apply in one
mode. An axis or normal of `None` (the default) follows the cell's
stack axis. Angles are in radians. `LayeredBiaxialOrientation` chooses
its directions per layer, so it must be combined with
`LayeredPosition`.

| Orientation | Meaning | Options |
| --- | --- | --- |
| `IsotropicOrientation()` | Uniform directions on the sphere. | no options |
| `PlanarOrientation(normal=None, max_tilt=...)` | Directions near the plane with this normal; `normal=None` uses the cell's stack axis. | `max_tilt` in rad |
| `LayeredBiaxialOrientation(primary_fraction=, cross_fraction=, max_in_plane_deviation=, max_tilt=, seed=)` | A primary in-plane direction, its transverse direction, and a random remainder, per layer. | fractions 0–1, angles in rad |
| `AlignedOrientation(axis, max_angle=...)` | Directions within a cone around `axis`. | axis letter/index or xyz vector; rad |

| Position | Meaning | Options |
| --- | --- | --- |
| `UniformPosition()` | Centers uniform in the cell. | no options |
| `LayeredPosition(layer_count, axis=None, jitter_fraction=...)` | Centers on evenly spaced planes; each plane becomes a `formation_layer`. | count; axis defaults to the stack axis; jitter relative to spacing |
| `DensityGradientPosition(axis=None, exponent=..., toward_high=...)` | Center density rises along one axis. | positive exponent; boolean |

In [ ]:
# Keyword arguments describe the whole population in one expression.
population = tangle.FiberPopulation(
    material=tangle.Material("fine", diameter=7 * um, min_bend_radius=35 * um),
    count=100,
    segments_per_fiber=8,
    seed=42,
    length=(0.2 * mm, 0.3 * mm),
    # Sample diameters between the fine and coarse sizes; None would
    # use the material's diameter for every fiber.
    diameter=(7 * um, 19 * um),
    curvature_amplitude=(0.0, 10 * um),
    # 40% near a primary in-plane direction, 40% near its transverse
    # direction, and the remaining 20% random. The plane normal
    # defaults to the stack axis (z).
    orientation=tangle.LayeredBiaxialOrientation(
        primary_fraction=0.4,
        cross_fraction=0.4,
        max_in_plane_deviation=math.radians(10),
        max_tilt=math.radians(5),
        seed=3,
    ),
    # Four evenly spaced planes along z become formation layers 0-3.
    position=tangle.LayeredPosition(4, jitter_fraction=0.2),
    max_attempts_per_fiber=256,
)
generated = tangle.generate_fiber_population(cell, population, name="four plies")
print(len(generated), generated.layer_ids())

In [ ]:
# replace() returns a modified copy, so variants share every other field.
variants = {
    "isotropic, uniform": population.replace(
        orientation=tangle.IsotropicOrientation(),
        position=tangle.UniformPosition(),
    ),
    "planar felt": population.replace(
        orientation=tangle.PlanarOrientation(max_tilt=math.radians(5)),
    ),
    "aligned with x": population.replace(
        orientation=tangle.AlignedOrientation("x", max_angle=math.radians(15)),
        length=0.25 * mm,
    ),
    # Layered-biaxial orientation needs layers, so pair the density
    # gradient with a planar orientation instead.
    "denser toward high z": population.replace(
        orientation=tangle.PlanarOrientation(max_tilt=math.radians(5)),
        position=tangle.DensityGradientPosition(exponent=2.0, toward_high=True),
    ),
}
{name: len(tangle.generate_fiber_population(cell, variant)) for name, variant in variants.items()}